In [1]:
import pandas as pd
import sqlite3
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))
from config import DB_FILE

conn = sqlite3.connect(DB_FILE)
print("Connected")


Done!
Connected


I need to create an index so SQLite works faster. Without an index, it is like finding every mention of "Cost of Sales" on every single page of a book. With it, it jumps straight to the right pages.

In [2]:
conn.execute("CREATE INDEX IF NOT EXISTS idx_GL_account  ON GL (Account_key)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_GL_date     ON GL (Date)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_GL_territory ON GL (Territory_key)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_Calendar_date ON Calendar (Date)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_COA_account ON COA (Account_key)")
conn.commit()
print("Indexes created")

Indexes created


First we will connect GL to COA and Calendar and filter by Profit and Loss. I group the amounts by year and the full COA hierachy.

In [ ]:
pl = pd.read_sql("""
    SELECT
        Calendar.Year,
        Calendar.Quarter,
        Calendar.Month,
        Class,
        SubClass,
        Account,
        SubAccount,
        SUM (Amount) AS Amount
    FROM gl
    JOIN coa ON GL.Account_key = GL.Account_key
    JOIN calendar ON Calendar.Date = calendar.Date
    WHERE Report = 'Profit and Loss'
    GROUP BY
        Calendar.Year,
        Calendar.Quarter,
        Calendar.Month,
        Class,
        SubClass,
        Account
    ORDER BY
        Calendar.Year,
        Calendar.Quarter,
        Calendar.Month,
        Class
""", conn) 

print(f"Rows returned: {len(pl)}")
pl.head(10)

We shall look for a positive amountof Sales. Cost of Sales of Sales and Expanses should be negative.

Now I simplify everything to create the classic P&L structure (Sales, Cost of Sales, Operating Expenses, etc., over your three-year period).

In [ ]:
annual = pd.read_sql("""
    SELECT
        cal.Year,
        coa.Class,
        coa.SubClass,
        SUM(gl.Amount) AS Amount
    FROM gl
    JOIN coa      ON gl.Account_key = coa.Account_key
    JOIN calendar  ON gl.Date        = calendar.Date
    WHERE coa.Report = 'Profit and Loss'
    GROUP BY cal.Year, coa.Class, coa.SubClass
    ORDER BY cal.Year, coa.Class, coa.SubClass
""", conn)
...

Results

The actual P&L Statement: I take the annual summary (from above) to compute it with the key P&L subtotals (Gross Profit, Operating Profit and Net Profit - for each year)

In [ ]:
waterfall = annual.groupby(['Year', 'SubClass'])['Amount'].sum().unstack('SubClass').fillna(0)

# Calculate P&L lines
waterfall['Revenue']          = waterfall.get('Sales', 0)
waterfall['Gross Profit']     = waterfall['Revenue'] + waterfall.get('Cost of Sales', 0)
waterfall['EBIT']             = waterfall['Gross Profit'] + waterfall.get('Operating Expenses', 0) + waterfall.get('Depreciation & Amortization', 0)
waterfall['Net Profit']       = waterfall['EBIT'] + waterfall.get('Interest Income', 0) + waterfall.get('Interest Expense', 0) + waterfall.get('Taxation', 0)

# Show only the key lines
summary = waterfall[['Revenue', 'Gross Profit', 'EBIT', 'Net Profit']]
print(summary)


Gross Profit should be less than Revenue <br>
EBIT should be less than Gross Profit <br>
Net Profit should be the smallest number <br>